# Testing hyperparameters of the model and training

In this notebook, we will discuss the importance of choosing the appropriate parameters for a CE model. We will not cover everything here, this is meant to serve as a discussion regarding which parameters to watch out for.

In [ ]:
# In this block, we do the same thing as for the training to set up
from icet import ClusterSpace, StructureContainer
from trainstation import CrossValidationEstimator
from pathlib import Path
import ase.io
from tqdm import tqdm


# This sets the sublattices in the CE model, multiple lists can be given for different sublattices at different sites
chemical_symbols = [["Cu", "Au"]]

# this specifies which dataset we are using by providing the maximum number of atoms to train on
max_atom_num = 8

base_path = Path.cwd().parents[1]
struct_path = (
    base_path
    / "data"
    / f"CE_dataset_{chemical_symbols[0][0]}{chemical_symbols[0][1]}"
)
dataset_path = (
    struct_path 
    / f"enumerated_structures_{max_atom_num}_calculated.extxyz"
)


primitive_path = (
    struct_path
    / f"{chemical_symbols[0][0]}_relaxed.extxyz"
)

# load the relaxed primitive structure from which we generated the dataset
prim = ase.io.read(primitive_path)


# we create a cluster space with certain cutoffs for pairs, triplets and quadruplets
cs = ClusterSpace(
    structure=prim,
    cutoffs=[13, 7, 6],
    chemical_symbols=chemical_symbols,
)

# the structure container needs to know the basic conditions of the cluster space
# the dataset will then be added based on the basic lattice
sc = StructureContainer(cluster_space=cs)

dataset = ase.io.read(dataset_path, index=":")
# pristine_structs = ase.io.read(pristine_dataset_path, index=":")

for aid, atoms in tqdm(enumerate(dataset)):
    # Note: the structures need positions and cell parameters to align to the primitive structure
    # we can use map_structure_to_reference to map the structure to the primitive
    # in our case, the positions are already on-lattice
    # atoms, info = map_structure_to_reference(atoms, prim)
    mapped = atoms
    sc.add_structure(
        atoms, properties=dict(mixing_energy=atoms.info["mixing_energy"])
    )

## Testing the regularization parameter

The $\lambda$ parameter of the arder algorithm (see the corresponding [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ARDRegression.html) documentation) can make quite a few differences in the results and there are a number of quantities that are reasonable to investigate. Their roles are:
- RMSE: representative for the error of the model. It is tempting to just use the minimum. However, the lowest value can be problematic as more parameters should naturally lead to a better agreement. If we have to many parameters we can encounter overfitting issues.
- $R^2$: The coefficient of determination provides a proportional value for the predicted and reference outcomes. Ideally it amounts to 1 and can represent a less dataset-dependent metric to assess model quality.
- BIC: The [Bayesian Information Criterion](https://en.wikipedia.org/wiki/Bayesian_information_criterion) is a commonly used tool to select optimal models in information theory. It helps to prevent overfitting when a model with a low BIC value is chosen.
- non-zero parameters: A sparse model is faster and commonly also more transferable compared to one containing a larger number of parameters. Its number is strongly dependent on the investigated system and the used regularization parameter.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

seed = np.random.randint(0, 1e5)
print("seed: ", seed)
valid = []
BICs = []
R2s = []
num_params = []
lambda_values = np.logspace(2, 5, 10)
for lbd in lambda_values:
    opt = CrossValidationEstimator(
        fit_data=sc.get_fit_data(key="mixing_energy"),
        fit_method="ardr",
        threshold_lambda=lbd,
        seed=seed,
    )
    opt.validate()
    opt.train()
    valid.append(opt.rmse_validation)
    BICs.append(opt.BIC)
    R2s.append(opt.R2_validation)
    num_params.append(opt.n_nonzero_parameters)
fig, axs = plt.subplots(2, 2, sharex=True)

min_bic = np.argmin(BICs)

plt.sca(axs[0, 0])
plt.plot(lambda_values, valid)
plt.axhline(opt.rmse_validation, color="r", linestyle="--")
plt.axvline(lambda_values[min_bic], color="k", linestyle="--")
plt.xlabel("r$\lambda$")
plt.ylabel(r"RMSE")
plt.sca(axs[0, 1])
plt.plot(lambda_values, R2s)
plt.axhline(opt.R2_validation, color="r", linestyle="--")
plt.axvline(lambda_values[min_bic], color="k", linestyle="--")
plt.xlabel(r"$\lambda$")
plt.ylabel(r"R$^2$")
plt.sca(axs[1, 0])
plt.plot(lambda_values, BICs)
plt.axhline(opt.BIC, color="r", linestyle="--")
plt.axvline(lambda_values[min_bic], color="k", linestyle="--")
plt.xlabel(r"$\lambda$")
plt.ylabel("BIC")
plt.sca(axs[1, 1])
plt.plot(lambda_values, num_params)
plt.axhline(opt.n_nonzero_parameters, color="r", linestyle="--")
plt.axvline(lambda_values[min_bic], color="k", linestyle="--")
plt.xlabel(r"$\lambda$")
plt.ylabel("non-zero parameters")
plt.tight_layout()
